# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 1: Basics of Prompt Engineering</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know that a language model continues one long sequence of tokens, <br>
        ... can write zero-, one-, few-shot, and chain-of-thought prompts for a base model, <br>
        ... know the three reasons why a generation stops, <br>
        ... can build a prompt template and measure how well it works.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

Documentation for this notebook:

* [Hugging Face: Generation strategies](https://huggingface.co/docs/transformers/generation_strategies) (decoding)
* [Ollama API documentation](https://docs.ollama.com/api)

## From Session 3 to the Continuation Machine

**In short: a language model gets the beginning of a text and continues it.**

The four steps from Session 3 ([3.01 - Writing with Transformers](https://github.com/olivermueller/amlta-2026/blob/main/Session_03/3.01%20-%20Writing%20with%20Transformers.ipynb)):

1. **Tokenization:** text becomes token IDs.
2. **Next-token prediction:** every token gets a probability.
3. **Decoding:** a rule picks one token ([overview](https://huggingface.co/docs/transformers/generation_strategies)).
4. **Autoregressive generation:** append the token and repeat.

Think of the model as a **continuation machine**. Everything today builds on this idea.

But first, we set up **Ollama**.

## Setup: Ollama

[Ollama](https://ollama.com/) is a free tool that runs language models on your own computer. Your code sends prompts to its local server. No data leaves your machine.

### Step 1: Install Ollama

* **Linux / Colab:** run the cell below (the installer needs `pciutils` and `zstd`).
* **macOS / Windows:** install the app from [ollama.com/download](https://ollama.com/download) and skip this cell.

In [ ]:
!sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

### Step 2: Start the Ollama server

* **Colab:** the installer may say that the server is running, but it isn't. So we start it ourselves.
* **macOS / Windows:** the app starts the server for you.

In [ ]:
import subprocess
import time

# the server writes its log into a file; a pipe that nobody reads could fill up and block the server
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)

# give the server a few seconds to start
time.sleep(5)

### Step 3: Pull the model

We download ("pull") **Llama 3.2 1B**. The `text` in its name means that this is a **base model**: it was only trained to continue text. What that means is the topic of this notebook.

In [ ]:
# the download takes about a minute
!ollama pull llama3.2:1b-text-q4_K_M

BASE_MODEL = "llama3.2:1b-text-q4_K_M"  # base model: only continues text

Strictly speaking, this is not a *large* language model: it has about 1.2 billion parameters, while large ones have tens to hundreds of billions. Models of this size are **small language models (SLMs)** and run on a laptop ([Wang et al., 2025](https://doi.org/10.1145/3711896.3736563)). We use it for learning only ;)

### Step 4: Connect from Python

We install the `ollama` package and connect to our local server. As a quick check, the server lists the model we just pulled.

**Why not Hugging Face, as in Session 3?**

* **Ollama and vLLM** are built for **inference**, i.e. for using a model: fast, memory-efficient, and for many users at once.
* **Hugging Face `transformers`** is built for looking inside a model and for training it. We come back to it when we fine-tune models.

In [ ]:
#!pip install ollama

import ollama

olm = ollama.Client(host="http://127.0.0.1:11434")  # address of our local Ollama server
for model in olm.list().models:
    print(model.model)

## Part 1: The Continuation Machine

**In short: a base model sees exactly your text and continues it.** No formatting, no roles, no special tokens.

We send text with `olm.generate()`. Two options control the decoding:

* `num_predict`: the maximum number of new tokens
* `temperature`: `0` means greedy decoding

Let's give it the beginning of a poem ("Twinkle, Twinkle, Little Star", Jane Taylor, 1806).

In [ ]:
poem = """Twinkle, twinkle, little star,
How I wonder what you are!
Up above the world so high,
Like a diamond in the sky.

When the blazing sun is gone,
When he nothing shines upon,
Then you show your little light,
Twinkle, twinkle, all the night.

"""
response = olm.generate(model=BASE_MODEL, prompt=poem, options={"num_predict": 60, "temperature": 0})
print(response.response)

The model continues the poem with the same line length and rhythm. It doesn't look anything up. It just writes a *plausible* continuation, token by token.

The response contains more than the text:

In [ ]:
print("tokens in the prompt:", response.prompt_eval_count)
print("tokens generated:", response.eval_count)
print("stopped because of:", response.done_reason)

`length` means that the model hit the `num_predict` limit. More on this in Part 4.

Did Ollama change our text before the model saw it? Every Ollama model has a **template** that wraps the prompt. For a base model, the template is just `{{ .Prompt }}`: your prompt, with nothing added.

In [ ]:
print(olm.show(BASE_MODEL).template)

### Your turn

**Predict first:** which of the two prompts gives you an answer about Einstein? And what does the model do with the other one?

In [ ]:
prompts = ["What were the main achievements of Albert Einstein?", "The main achievements of Albert Einstein were"]

for prompt in prompts:
    response = olm.generate(model=BASE_MODEL, prompt=prompt, options={"num_predict": 80, "temperature": 0})
    print(prompt + response.response)
    print("-" * 40)

<details>
<summary><b>Explanation</b> (open after running the cells)</summary>

* **Prompt 1** is a question. A base model does not answer questions. It treats the question like the title of a document and continues with whatever could follow: a list, more questions, or something unrelated.
* **Prompt 2** is the *beginning of an answer*, so the continuation reads like an answer ([Liu et al., 2023](https://doi.org/10.1145/3560815)).
* Check the facts: fluent text is not the same as true text.
</details>

## Part 2: Prompting a Base Model: Zero-, One-, and Few-Shot

**In short: write the beginning of a text whose most likely continuation is your answer.**

* **Prompting:** give an LLM a task as text, without retraining it.
* **Prompt (template) engineering:** find the prompt that solves the task best.

([Kaltenpoth and Müller, 2024](https://aisel.aisnet.org/wi2024/91/); [Liu et al., 2023](https://doi.org/10.1145/3560815))

The simplest way: just ask, without examples. This is **zero-shot prompting** ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)).

In [ ]:
question = """When did world war 2 start?"""

qa_prompt = """Q: When did world war 2 start?
A:"""

for prompt in [question, qa_prompt]:
    response = olm.generate(model=BASE_MODEL, prompt=prompt, options={"num_predict": 40, "temperature": 0})
    print(prompt + response.response)
    print("-" * 40)

The question alone rarely gives a clean answer. The `Q:`/`A:` format works better, because in the training data an `A:` usually follows a `Q:`.

**A small detail:** the prompt ends with `A:`, not `A: `. Tokenizers usually attach the space to the *next* word (` 1939` is one token). A trailing space would give the model an unusual token sequence.

Next, we classify the sentiment of a movie review: first without an example, then with one. A prompt with one example is called **one-shot prompting** ([Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)).

In [ ]:
zero_shot_prompt = """'I love this movie' is a"""

one_shot_prompt = """'I hate this movie' is a negative comment.
'I love this movie' is a"""

for prompt in [zero_shot_prompt, one_shot_prompt]:
    response = olm.generate(model=BASE_MODEL, prompt=prompt, options={"num_predict": 20, "temperature": 0})
    print(prompt + response.response)
    print("-" * 40)

With several examples, it is called **few-shot prompting** ([Liu et al., 2023](https://doi.org/10.1145/3560815); [Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)). Here with the `Q:`/`A:` format:

In [ ]:
few_shot_prompt = """Q: What is 11x2/2? A: 11
Q: What is 11x4/2? A: 22
Q: What is 11x6/2? A: 33
Q: What is 11x8/2? A:"""

response = olm.generate(model=BASE_MODEL, prompt=few_shot_prompt, options={"num_predict": 40, "temperature": 0, "seed": 42})
print(response.response)

### Your turn

1. Look at the output of the few-shot cell again. **What does the model write after the answer, and why?**

<details>
<summary><b>Explanation for question 1</b></summary>

The model keeps going and invents new questions and answers. The most likely continuation of a list of `Q:`/`A:` pairs is another pair, and nothing in the text tells it to stop. Remember this for Part 4.
</details>

2. **A real task: sentiment classification** of ten movie reviews.

    * A **prompt template** is a prompt with a placeholder, here `{text}`.
    * `score()` fills in every review, asks the model, and counts the correct answers. It also prints the first prompt, so you see what the model sees.

    Your steps:

    * Run the next cell to define `score()`.
    * Run the zero-shot template and read the answers.
    * Write a one-shot and a few-shot template with your own example reviews. Don't use reviews from the list, otherwise the model can just copy the answer.
    * Which template scores best? Compare with your neighbours.

In [ ]:
import re

def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def score(template, examples, max_tokens=10, stop=None):
    """Fills {text} in the template with every example, sends it to the model, and counts the correct answers."""
    print("Prompt for the first example:\n" + template.replace("{text}", examples[0][0]) + "\n")
    options = {"num_predict": max_tokens, "temperature": 0}
    if stop:
        options["stop"] = stop
    correct, tokens = 0, 0
    for text, expected in examples:
        response = olm.generate(model=BASE_MODEL, prompt=template.replace("{text}", text), options=options)
        ok, answer = check(response.response, expected)
        correct += ok
        tokens += response.eval_count or 0  # an empty prompt generates nothing
        print(f"{'✓' if ok else '✗'} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {tokens} tokens generated")

In [ ]:
# ten short movie reviews with their sentiment
REVIEWS = [
    ("A wonderful film with brilliant actors.", "positive"),
    ("The plot was boring and far too long.", "negative"),
    ("I laughed from start to finish.", "positive"),
    ("A complete waste of two hours.", "negative"),
    ("Not bad at all, I would watch it again.", "positive"),
    ("The ending made no sense, and the dialogue was awful.", "negative"),
    ("Beautiful music and a touching story.", "positive"),
    ("I fell asleep halfway through.", "negative"),
    ("It was not as good as everyone said.", "negative"),
    ("My kids loved it, and so did I.", "positive"),
]

In [ ]:
zero_shot = """Review: {text}
Sentiment:"""
score(zero_shot, REVIEWS)

In [ ]:
# TODO: write a one-shot and a few-shot template, and keep the placeholder {text}
one_shot = """Review: The story was dull and the actors were wooden.
Sentiment: negative

Review: {text}
Sentiment:"""
few_shot = """Review: The story was dull and the actors were wooden.
Sentiment: negative

Review: What a great surprise, I enjoyed every minute.
Sentiment: positive

Review: The jokes fell flat and I left early.
Sentiment: negative

Review: A charming film that the whole family will like.
Sentiment: positive

Review: {text}
Sentiment:"""

score(one_shot, REVIEWS)
score(few_shot, REVIEWS)

## Part 3: From Few-Shot Prompts to Conversations

**In short: write every task as a conversation between a user and an assistant.**

Inventing a new text pattern for every task is tedious. A conversation works as one pattern for all tasks, and a base model has seen lots of them: forums, interviews, chat logs.

The same trick can show the model *how* to answer. In **chain-of-thought (CoT) prompting**, the example answers contain the reasoning steps before the result ([Wei et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf)):

![Chain-of-Thought Prompting](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_04/imgs/cot.png)

(Figure from [Wei et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf).) Note that this is still few-shot prompting: the examples show the model how to reason.

The example from the paper, written as a conversation:

In [ ]:
conversation = """The following is a conversation between a user and a helpful assistant. The assistant thinks step by step before it answers.

User: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
Assistant: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.

User: The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?
Assistant:"""

response = olm.generate(model=BASE_MODEL, prompt=conversation, options={"num_predict": 150, "temperature": 0, "seed": 42})
print(response.response)

Look at the end of the output. The model probably didn't stop after its answer. It also wrote the next `User:` line, maybe even a whole new question.

The most likely continuation of a conversation is more conversation, so the base model plays **both roles**.

### Your turn

Build a chain-of-thought prompt for the six word problems below.

1. Write a conversation template with at least one step-by-step example of your own.
    * End the example answer with `The answer is ...`, because `score()` looks for the answer there.
    * End the template with `User: {text}` and an open `Assistant:`.
2. Copy the template and remove the reasoning from the example, so only `The answer is ...` remains. Which version scores better?

Don't be surprised if it goes wrong: small models often copy the *form* of the steps and still get the arithmetic wrong.

In [ ]:
# six word problems with their answers
WORD_PROBLEMS = [
    ("A baker makes 4 trays with 6 cookies on each tray. He sells 10 cookies. How many cookies are left?", "14"),
    ("Lisa has 3 boxes with 5 pencils each. She gives 4 pencils to her brother. How many pencils does she have now?", "11"),
    ("A bus has 20 passengers. At the first stop, 7 get off and 3 get on. How many passengers are on the bus now?", "16"),
    ("Tom reads 8 pages every day. How many pages does he read in one week?", "56"),
    ("A shirt costs 15 euros. Anna buys 2 shirts and pays with a 50 euro note. How much change does she get?", "20"),
    ("There are 12 eggs in a carton. Mia uses 5 eggs for a cake and 2 for breakfast. How many eggs are left?", "5"),
]

In [ ]:
# TODO 1: your conversation template with a step-by-step example
cot_template = """The following is a conversation between a user and a helpful assistant. The assistant thinks step by step before it answers.

User: Sara has 2 bags with 9 apples each. She eats 5 apples. How many apples does she have left?
Assistant: 2 bags with 9 apples each are 2 * 9 = 18 apples. She eats 5, so 18 - 5 = 13 are left. The answer is 13.

User: {text}
Assistant:"""
score(cot_template, WORD_PROBLEMS, max_tokens=150)

# TODO 2: the same template without the reasoning steps
no_cot_template = """The following is a conversation between a user and a helpful assistant.

User: Sara has 2 bags with 9 apples each. She eats 5 apples. How many apples does she have left?
Assistant: The answer is 13.

User: {text}
Assistant:"""
score(no_cot_template, WORD_PROBLEMS, max_tokens=150)

## Part 4: How Does the Model Know When to Stop?

**In short: a language model never "decides" that it is done. Generation stops for one of three reasons:**

1. The **maximum number of tokens** is reached (`num_predict` in Ollama, `max_tokens` in most other APIs). Ollama then reports `done_reason` as `length`.
2. The model generates a **special end token**. Ollama reports `stop`. A base model knows `<|end_of_text|>` for the end of a document, but it rarely predicts it in the middle of a conversation.
3. The application finds a **stop sequence** in the text and cuts it off there. This is also reported as `stop`.

Let's cut the text as soon as the model starts the next user turn:

In [ ]:
conversation = """The following is a conversation between a user and a helpful assistant. The assistant thinks step by step before it answers.

User: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
Assistant: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.

User: The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?
Assistant:"""

response = olm.generate(
    model=BASE_MODEL,
    prompt=conversation,
    options={"num_predict": 150, "temperature": 0, "stop": ["\nUser:"]},
)
print(response.response)
print("stopped because of:", response.done_reason)

The model still *wants* to write the user's line; Ollama just cuts it off. That is a crude fix. What happens if a user writes "User:" in their own message?

**The better fix needs training:** teach the model to end every answer with a special **end-of-turn token**, and stop as soon as it generates it. That is exactly what chat models learn.

### Your turn

1. Run the few-shot math prompt from Part 2 again with `stop=["\n"]`. **Predict** the output and the `done_reason`.
2. Score your best sentiment template again with `stop=["\n"]`. The accuracy should stay the same. Compare the answers and the number of tokens. Why is this useful in an application?
3. You probably know ChatGPT. How do its answers differ from those of our base model? Write your answer as a comment below and share it with the seminar.

In [ ]:
# TODO 1: the few-shot math prompt from Part 2 (few_shot_prompt) with a stop sequence
response = olm.generate(model=BASE_MODEL, prompt=few_shot_prompt, options={"num_predict": 40, "temperature": 0, "seed": 42, "stop": ["\n"]})
print(response.response)
print("stopped because of:", response.done_reason)  # stop: the line break ended the generation, not the token limit

# TODO 2: your best sentiment template from Part 2, now with a stop sequence
score(few_shot, REVIEWS, stop=["\n"])

In [ ]:
# Write here your answer to the question: What is different in this model's answers compared to those of ChatGPT?
#
# ChatGPT answers the question and then stops. Our base model only continues the text:
# - it does not answer a question, but continues it like the start of a document,
# - it does not stop after its answer, but invents the next question or the next "User:" line,
# - it only stops at the token limit or at a stop sequence that we set.
# ChatGPT is a chat model: it was fine-tuned on conversations and ends every answer with a special end token (-> 4.02).

## Questions?

Anything unclear about base models, few-shot patterns, or stopping? Ask now: the next notebook builds on it.

## What You Learned

* A language model sees **one sequence of tokens** and continues it.
* **Base model:** what you send is what the model sees.
* **Zero-, one-, few-shot** and **chain-of-thought** are patterns in the text. The model continues the pattern and does not stop on its own.
* Generation stops at the **maximum number of tokens**, a **stop sequence**, or a **special end token**.
* A **prompt template** has a placeholder for the input. With a few examples where you know the answer, you can measure which template works best.
* Chat models learn an **end-of-turn token** instead of a stop sequence.

Next up: **4.02 - Prompt Template Engineering**, with chat templates, the system prompt, and reasoning models.